# Other renderers 01: ytt

ytt (Carvel) templates YAML with Starlark annotations that respect the structure: values are data, overlays patch by matching nodes instead of text.


In [ ]:
export HOME=/tmp
mkdir -p /source/work/other-lab/ytt && cd /source/work/other-lab/ytt
cat > values.yml <<'YAML'
#@data/values
---
name: web
replicas: 1
image: traefik/whoami:v1.11.0
YAML
cat > deployment.yml <<'YAML'
#@ load("@ytt:data", "data")
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: #@ data.values.name
spec:
  replicas: #@ data.values.replicas
  selector:
    matchLabels:
      app: #@ data.values.name
  template:
    metadata:
      labels:
        app: #@ data.values.name
    spec:
      containers:
        - name: #@ data.values.name
          image: #@ data.values.image
YAML
ytt -f deployment.yml -f values.yml


In [ ]:
cd /source/work/other-lab/ytt
ytt -f deployment.yml -f values.yml -v replicas=3 -v name=web-prod | yq '.metadata.name, .spec.replicas'


Overlays are the structural patch: match a document and edit nodes. No indentation to get wrong.


In [ ]:
cd /source/work/other-lab/ytt
cat > prod-overlay.yml <<'YAML'
#@ load("@ytt:overlay", "overlay")
#@overlay/match by=overlay.subset({"kind": "Deployment"})
---
spec:
  #@overlay/match missing_ok=True
  replicas: 3
  template:
    spec:
      containers:
        #@overlay/match by="name"
        - name: web
          #@overlay/match missing_ok=True
          resources:
            requests: {cpu: 100m, memory: 64Mi}
YAML
ytt -f deployment.yml -f values.yml -f prod-overlay.yml | yq '.spec.replicas, .spec.template.spec.containers[0].resources'
